### Setup

In [1]:
# Data
import pandas as pd
import numpy as np

# Sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def log(msg):
    # Simple wrapper so you can later swap with logging.info()
    print(f"[INFO] {msg}")

log("Importing required libraries")

[INFO] Importing required libraries


### Loading Data

In [2]:
#I have downloaded the data set from https://www.kaggle.com/datasets/blastchar/telco-customer-churn
#I have renamed the file telco_churn.csv and set in the proper directory
data_path = "../data/raw/telco_churn.csv"
log("Loading data from " + data_path)
df = pd.read_csv(data_path)
log(f"Loaded data with shape {df.shape}")


[INFO] Loading data from ../data/raw/telco_churn.csv
[INFO] Loaded data with shape (7043, 21)


### Cleaning

In [11]:
def convert_to_numeric(df, columns):
    for col in columns:
        log(f"Converting {col} to numeric")
        df[col] = pd.to_numeric(df[col], errors='coerce')
        log(f"{col} dtype after conversion: {df[col].dtype}")

        if df[col].isna().any():
            log(f"WARNING: NaNs detected in {col} after conversion")
    return df


def map_binary_columns(df, column_map):
    for col, mapping in column_map.items():
        if df[col].dtype == 'object':
            log(f"Mapping binary column {col} with {mapping}")
            df[col] = df[col].map(mapping)
    return df


def safe_binary_from_condition(df, col, condition_fn, description=""):
    log(f"Applying binary transform on {col} ({description})")
    df[col] = condition_fn(df[col]).astype(int)
    return df


def preserve_and_binarize(df, col, new_col_name, condition_fn):
    if new_col_name not in df.columns:
        log(f"Preserving original column {col} into {new_col_name}")
        df[new_col_name] = df[col]

    log(f"Binarizing {col}")
    df[col] = condition_fn(df[col]).astype(int)
    return df


def validate_and_binarize(df, columns, reference_col, invalid_value):
    for col in columns:
        log(f"Validating column {col} against {reference_col}")

        sanity_passed = (
            df.loc[df[col] == invalid_value, reference_col] == 0
        ).all()

        if not sanity_passed:
            raise ValueError(f"Sanity check failed for {col}, aborting transformation")

        log(f"Sanity check passed for {col}, converting to binary")
        df[col] = (df[col] == 'Yes').astype(int)

    return df


def one_hot_encode(df, columns):
    log(f"One-hot encoding columns: {columns}")
    # Make copies of the original columns
    for col in columns:
        df[f"{col}_orig"] = df[col]
    # One-hot encode (keeps original columns untouched)
    df = pd.get_dummies(df, columns=columns, prefix=columns, dtype=int)
    return df


def clean_dataframe(df):
    log("Starting data cleaning pipeline")

    # 1. Numeric conversions
    df = convert_to_numeric(df, ['TotalCharges'])

    # 2. Binary mappings
    binary_map = {
        'gender': {'Female': 1, 'Male': 0},
        'Partner': {'Yes': 1, 'No': 0},
        'Dependents': {'Yes': 1, 'No': 0},
        'PhoneService': {'Yes': 1, 'No': 0},
        'PaperlessBilling': {'Yes': 1, 'No': 0},
    }
    df = map_binary_columns(df, binary_map)

    # 3. Special binary transforms
    df = safe_binary_from_condition(
        df,
        'MultipleLines',
        lambda x: x == 'Yes',
        description="Yes vs everything else"
    )

    # 4. Internet service handling
    df = preserve_and_binarize(
        df,
        col='InternetService',
        new_col_name='InternetService_Category',
        condition_fn=lambda x: x != 'No'
    )

    # 5. Dependent internet features
    internet_features = [
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]

    df = validate_and_binarize(
        df,
        columns=internet_features,
        reference_col='InternetService',
        invalid_value='No internet service'
    )


    # 6. One-hot encoding
    df = one_hot_encode(
        df,
        ['Contract', 'PaymentMethod', 'InternetService_Category']
    )

    log("Data cleaning pipeline completed")

    return df

df = pd.read_csv(data_path)
df = clean_dataframe(df)

[INFO] Starting data cleaning pipeline
[INFO] Converting TotalCharges to numeric
[INFO] TotalCharges dtype after conversion: float64
[INFO] WARNING: NaNs detected in TotalCharges after conversion
[INFO] Mapping binary column gender with {'Female': 1, 'Male': 0}
[INFO] Mapping binary column Partner with {'Yes': 1, 'No': 0}
[INFO] Mapping binary column Dependents with {'Yes': 1, 'No': 0}
[INFO] Mapping binary column PhoneService with {'Yes': 1, 'No': 0}
[INFO] Mapping binary column PaperlessBilling with {'Yes': 1, 'No': 0}
[INFO] Applying binary transform on MultipleLines (Yes vs everything else)
[INFO] Preserving original column InternetService into InternetService_Category
[INFO] Binarizing InternetService
[INFO] Validating column OnlineSecurity against InternetService
[INFO] Sanity check passed for OnlineSecurity, converting to binary
[INFO] Validating column OnlineBackup against InternetService
[INFO] Sanity check passed for OnlineBackup, converting to binary
[INFO] Validating column

### Feature Engineering

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin

class DataCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        return clean_dataframe(X)

In [5]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_charge_ = X['MonthlyCharges'].median()
        return self

    def transform(self, X):
        X = X.copy()

        X['tenure_group'] = pd.cut(
            X['tenure'],
            bins=[0, 12, 24, 48, 72],
            labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr']
        )

        X['is_new_customer'] = (X['tenure'] <= 12).astype(int)

        X['is_long_term'] = (
            X['Contract_One year'] + X['Contract_Two year']
        ).astype(int)

        X['auto_pay_flag'] = (
            X['PaymentMethod_Bank transfer (automatic)'] +
            X['PaymentMethod_Credit card (automatic)']
        ).astype(int)

        service_cols = [
            'OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies'
        ]
        X['num_services'] = X[service_cols].sum(axis=1)

        X['avg_revenue'] = X['TotalCharges'] / (X['tenure'] + 1)

        X['high_monthly_flag'] = (
            X['MonthlyCharges'] > self.median_charge_
        ).astype(int)

        X['family_flag'] = (
            (X['Partner'] + X['Dependents']) > 0
        ).astype(int)

        X['fiber_flag'] = X['InternetService_Category_Fiber optic']
        X['electronic_check_flag'] = X['PaymentMethod_Electronic check']

        # interactions
        X['new_echeck_interaction'] = (
            X['is_new_customer'] * X['electronic_check_flag']
        )

        X['fiber_highcharge_interaction'] = (
            X['fiber_flag'] * X['high_monthly_flag']
        )

        X['loyal_engaged_interaction'] = (
            X['is_long_term'] * X['num_services']
        )

        return X

In [6]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_revenue']

engineered_features = [
    'is_new_customer', 'is_long_term', 'auto_pay_flag', 'num_services',
    'high_monthly_flag', 'family_flag', 'fiber_flag', 'electronic_check_flag',
    'new_echeck_interaction', 'fiber_highcharge_interaction', 'loyal_engaged_interaction'
]

categorical_features = [
    'Contract_orig', 'PaymentMethod_orig',
    'InternetService_Category_orig', 'tenure_group'
]

In [7]:
# -------------------------------
# Reusable pipeline steps
# -------------------------------
median_imputer = SimpleImputer(strategy='median')
most_frequent_imputer = SimpleImputer(strategy='most_frequent')
missing_cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')
scaler = StandardScaler()
ohe_drop_first = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
ohe_all = OneHotEncoder(handle_unknown='ignore', sparse_output=False)


# -------------------------------
# 1. Linear model preprocessing
# -------------------------------
num_pipeline_linear = Pipeline([
    ('imputer', median_imputer),
    ('scaler', scaler)
])

cat_pipeline_linear = Pipeline([
    ('imputer', missing_cat_imputer),
    ('encoder', ohe_drop_first)
])

linear_preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline_linear, numeric_features + engineered_features),
        ('cat', cat_pipeline_linear, categorical_features)
    ]
)

# -------------------------------
# 2. Tree-based preprocessing
# -------------------------------
num_pipeline_tree = Pipeline([
    ('imputer', median_imputer)
])

cat_pipeline_tree = Pipeline([
    ('imputer', most_frequent_imputer),
    ('encoder', ohe_all)
])

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline_tree, numeric_features + engineered_features),
        ('cat', cat_pipeline_tree, categorical_features)
    ]
)


full_pipeline_linear = Pipeline([
    ('cleaning', DataCleaner()),
    ('feature_engineering', FeatureEngineer()),
    ('preprocessing', linear_preprocessor)
])

full_pipeline_tree = Pipeline([
    ('cleaning', DataCleaner()),
    ('feature_engineering', FeatureEngineer()),
    ('preprocessing', tree_preprocessor)
])

In [8]:
df_raw = pd.read_csv(data_path)

X_raw = df_raw.drop(columns='Churn')
y = df_raw['Churn']
y = y.map({'Yes': 1, 'No': 0})

from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

# Transform
X_train_linear = full_pipeline_linear.fit_transform(X_train_raw, y_train)
X_test_linear  = full_pipeline_linear.transform(X_test_raw)

X_train_tree = full_pipeline_tree.fit_transform(X_train_raw, y_train)
X_test_tree  = full_pipeline_tree.transform(X_test_raw)

[INFO] Starting data cleaning pipeline
[INFO] Converting TotalCharges to numeric
[INFO] TotalCharges dtype after conversion: float64
[INFO] WARNING: NaNs detected in TotalCharges after conversion
[INFO] Mapping binary column gender with {'Female': 1, 'Male': 0}
[INFO] Mapping binary column Partner with {'Yes': 1, 'No': 0}
[INFO] Mapping binary column Dependents with {'Yes': 1, 'No': 0}
[INFO] Mapping binary column PhoneService with {'Yes': 1, 'No': 0}
[INFO] Mapping binary column PaperlessBilling with {'Yes': 1, 'No': 0}
[INFO] Applying binary transform on MultipleLines (Yes vs everything else)
[INFO] Preserving original column InternetService into InternetService_Category
[INFO] Binarizing InternetService
[INFO] Validating column OnlineSecurity against InternetService
[INFO] Sanity check passed for OnlineSecurity, converting to binary
[INFO] Validating column OnlineBackup against InternetService
[INFO] Sanity check passed for OnlineBackup, converting to binary
[INFO] Validating column

### Modelling

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score


# Instantiate baseline logistic regression
logreg = LogisticRegression(random_state=42, max_iter=1000)

# Fit on preprocessed linear training data
logreg.fit(X_train_linear, y_train)

# Predict
y_pred_logreg = logreg.predict(X_test_linear)

y_proba_logreg = logreg.predict_proba(X_test_linear)[:, 1]
# Evaluate
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_logreg))




# Instantiate baseline random forest
rf = RandomForestClassifier(random_state=42, n_estimators=100)

# Fit on tree-preprocessed data
rf.fit(X_train_tree, y_train)

# Predict
y_pred_rf = rf.predict(X_test_tree)
y_proba_rf = rf.predict_proba(X_test_tree)[:, 1]

# Evaluate
print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))



# Instantiate baseline gradient boosting
gb = GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.1)

# Fit on tree-preprocessed data
gb.fit(X_train_tree, y_train)

# Predict
y_pred_gb = gb.predict(X_test_tree)
y_proba_gb = gb.predict_proba(X_test_tree)[:, 1]

# Evaluate
print("Gradient Boosting")
print("Accuracy:", accuracy_score(y_test, y_pred_gb))

Logistic Regression
Accuracy: 0.7906316536550745
Random Forest
Accuracy: 0.772888573456352
Gradient Boosting
Accuracy: 0.7920511000709723


In [10]:
# Model Comparison

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

# Collect metrics in a dictionary
metrics = {
    "Model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_logreg),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_logreg),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_gb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_logreg),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_logreg),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_logreg),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb)
    ]
}

# Create a DataFrame for clean comparison
comparison_df = pd.DataFrame(metrics)
comparison_df

,Model,Accuracy,ROC-AUC,Precision,Recall,F1 Score
0,Logistic Regression,0.790632,0.841583,0.638596,0.486631,0.552352
1,Random Forest,0.772889,0.808657,0.588235,0.481283,0.529412
2,Gradient Boosting,0.792051,0.839347,0.631922,0.518717,0.569750
